In [ ]:
# Set up libraries
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.tree import plot_tree
from sklearn.pipeline import Pipeline
import numpy as np
import matplotlib.pyplot as plt
import joblib

In [ ]:
# Read the clean dataset as DataFrame
path = r"C:\Users\13926\Atlantic-Project\AmesHousingClean.csv"
df = pd.read_csv(path)

In [ ]:
# Split the data into features and target and into training and testing datasets
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

In [ ]:
# Scale using Standard Scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Do grid-serach to find the best hyperparameters to plug in Random Forest Regressor
param_grid = {
    'n_estimators': [100, 150, 200, 250, 300],
    'max_features': ['sqrt', 'log2', 1.0],
    'max_depth': [10, 20, 30, None],
    'min_samples_leaf': [1, 2, 4],
    'min_samples_split': [2, 5, 10],
}

random_search = RandomizedSearchCV(RandomForestRegressor(random_state=42), 
                                   param_distributions=param_grid, 
                                   n_iter=30,
                                   cv=5,
                                   scoring='neg_mean_absolute_error',
                                   n_jobs=-2, 
                                   random_state=42)
random_search.fit(X_train_scaled, y_train)
print('Best Parameters', random_search.best_params_)

Best Parameters {'n_estimators': 150, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 10}


In [ ]:
rf_model = RandomForestRegressor(**random_search.best_params_)
rf_model.fit(X_train_scaled, y_train)
y_train_pred_rf = rf_model.predict(X_train_scaled)
y_pred_rf = rf_model.predict(X_test_scaled)

mae_train = mean_absolute_error(y_train, y_train_pred_rf)
mae_test = mean_absolute_error(y_test, y_pred_rf)
mape_train = mean_absolute_percentage_error(y_train, y_train_pred_rf) * 100
mape_test = mean_absolute_percentage_error(y_test, y_pred_rf) * 100
r2_train = r2_score(y_train, y_train_pred_rf)
r2_test = r2_score(y_test, y_pred_rf)

print(f'MAE (training): {mae_train}')
print(f'MAE (testing): {mae_test}')
print(f'MAPE (training): {mape_train:.4f}%')
print(f'MAPE (testing): {mape_test:.4f}%')
print(f'R-Squared score (training): {r2_train}')
print(f'R-Squared score (testing): {r2_test}')

MAE (training): 11522.470890753639
MAE (testing): 16800.258742742208
MAPE (training): 6.8249%
MAPE (testing): 9.6746%
R-Squared score (training): 0.9572811096953588
R-Squared score (testing): 0.8844831524463169


In [ ]:
# The data is outdated so we need an inflation factor so that the model also works when dealing with new data
inflation_factor = 1.867

In [ ]:
pipe = Pipeline([('scaler', StandardScaler()),
                 ('randforest', RandomForestRegressor(n_estimators=150,
                                                       min_samples_split=5,
                                                       min_samples_leaf=1,
                                                       max_features='sqrt',
                                                       max_depth=10))])
pipe.fit(X_train_scaled, y_train)
y_train_pred_rf = pipe.predict(X_train_scaled)
joblib.dump(pipe, "random_forest_pipeline.pkl")

['random_forest_pipeline.pkl']

In [ ]:
pipe = Pipeline([('scaler', StandardScaler()),
                 ('randforest', RandomForestRegressor(n_estimators=150,
                                                       min_samples_split=5,
                                                       min_samples_leaf=1,
                                                       max_features='sqrt',
                                                       max_depth=10))])
pipe.fit(X_train_scaled, y_train)
y_train_pred_rf = pipe.predict(X_train_scaled)
joblib.dump(pipe, "random_forest_pipeline.pkl")

['random_forest_pipeline.pkl']